In [ ]:
import os

import nibabel as nib
import numpy as np
from pydicom import dcmread
from pydicom.pixels import apply_rescale
from pmi_viewer import view

# Assignments 'Functions'

In this assignment, you will write functions for working with DICOM and Nifti files. Note that you only have to make changes to the functions in the cell below! Each time you make a change, you'll need to execute that cell again to test your new code.

*Hint: When adding a new feature to function, make a copy of the previous version and comment it out using hash-symbols (`#`). You can comment or uncomment a multiple lines at once by selecting them and pressing `Ctrl+/`*.

Now scroll down to the first cell that says **Assignment**, and start from there.

In [ ]:
def get_z_position(dicom_image):
    """
    Returns the z-coordinate of the Image Position Patient DICOM attribute.
    """
    return dicom_image.ImagePositionPatient[2]


def get_spacing(dicom_list):
    n = len(dicom_list)
    dx, dy = dicom_list[0].PixelSpacing
    
    if n > 1:
        z_first = np.array(dicom_list[0].ImagePositionPatient)
        z_last = np.array(dicom_list[-1].ImagePositionPatient)
        dz = np.linalg.norm(z_last - z_first) / (n - 1)
        return [dx, dy, dz]
    else:
        return [dx, dy]


def read_nifti(file_path):
    """
    Reads a Nifti file and returns a the data.
    """
    nifti_image = nib.load(file_path)  # Read the image.
    return nifti_image.get_fdata()

> **Assignment**
>
> **Add a `read_dicom` function**: Add this function to the cell above. It should take a file path and return a Numpy array with physical values (e.g. Hounsfield units for CT). The code below should now work without any modifications.
>
> Move your cursor over the air around the head. *Are the CT values close to -1000 HU?*

In [ ]:
data = read_dicom('images/ct_jaw_slice.dcm')
view(data)

> **Assignment**
>
> Now we'll test the given `read_nifti` function. Unlike DICOM, the Nifti file format allows using floating point numbers or signed integers for storing physical pixel values, so we don't need to convert the stored pixel values.
>
> **Now run the code in the cell below**: *Why is the orientation wrong?* The Nifti format and the `nibabel` package assume [RAS orientation](https://slicer.readthedocs.io/en/latest/user_guide/coordinate_systems.html#anatomical-coordinate-system) and [column-major ordering](https://en.wikipedia.org/wiki/Row-_and_column-major_order), whereas our the `view` function assumes LPS orientation and row-major ordering.
>
> **Fix the `read_nifti` function**: Use [numpy.transpose](https://numpy.org/doc/stable/reference/generated/numpy.transpose.html) to fix the ordering. Use [numpy.flip](https://numpy.org/doc/stable/reference/generated/numpy.flip.html) to mirror the axes (right-left and anterior-posterior). Check whether the displayed image looks exactly like the image in the cell above.

In [ ]:
data = read_nifti('images/ct_jaw_slice.nii.gz')
view(data)

> **Assignment**
>
> So far, we've been working with 2D DICOM images, but medical images are often 3D or even 4D (time series). Let's add a new *feature* to `read_dicom`:
>
> **Make `read_dicom` able to read directories with DICOM files**. If a directory is provided to `read_dicom`, the function should iterate over all files in the directory, read the DICOM images, and read and return a 3D array. If a file name is provided, `read_dicom` should continue to read the file as before.
> 
> *Hint 1: Use `os.path.isfile()` to check whether the path refers to a file and `os.path.isdir()` to check whether it refers to a directory. Use `os.listdir()` obtain a list of items in a directory. For example:*
> ```python
> path_name = 'images/ct_jaw'
> for file_name in os.listdir(path_name):  # Iterate over all items in path_name.
>      full_name = os.path.join(path_name, file_name)  # Join the directory name and item name using the system path separator (\ or /).
>      if os.path.isfile(full_name):  # Check if the item is a file. If it is, print its path.
>          print(full_name)
> ```
>
> *Hint 2: Append the pixel data to a list, then convert that list to a NumPy array using `np.array(my_list)`.*
>
> **Now run the code in the cell below and scroll through the slices**. *Why are the slices displayed in random order?*
>
> **Make `read_dicom` sort the slices**. Append the result from `dcmread()` to a list and use the function [sorted()](https://www.w3schools.com/python/ref_func_sorted.asp) to sort that list using `get_z_position` as a `key` function. Run the cell below again and check if the slices are now in correct order.

In [ ]:
data = read_dicom('images/ct_jaw')
view(data)

> **Assignment**
>
> We also need to read the pixel spacing and slice spacing in order to display images with *anisotropic* spacing correctly.
>
> **Run the code in the first cell below**. The keyword argument `orientation='sag'` configures the viewer to show the *sagittal* [anatomical plane](https://en.wikipedia.org/wiki/Anatomical_plane). *But this sagittal view doesn't look right!* Without specifying the spacing, the viewer assumes isotropic voxels.
>
> **Add a new feature to `read_dicom`**: Make the function return both the data and the spacing. Use the provided function `get_spacing()` to calculate the spacing given a list of results from `dcmread`.
>
> **Run the code in the second cell below**. *Does it look better now?* The code in this cell should work without any modifications. If it doesn't, revise your changes to `read_dicom` accordingly. *Note: the code in the first cell no longer works because of these changes, but that's fine.*
>
> **Add comments to `get_spacing`**: Read the function and add a docstring and inline comments where necessary.

In [ ]:
data = read_dicom('images/ct_jaw')
view(data, orientation='sag')

In [ ]:
data, spacing = read_dicom('images/ct_jaw')
view(data, spacing=spacing, orientation='sag')

**Supplementary information**: *You might just wonder why `get_spacing()` uses the '[Image Position (Patient)](https://dicom.innolitics.com/ciods/ct-image/image-plane/00200032)' DICOM attribute instead of '[Spacing Between Slices](https://dicom.innolitics.com/ciods/ct-image/image-plane/00180088)'. This is because the latter is optional, whereas the first is a required attribute. We can't simply rely on 'Spacing Between Slices' being present.*

> **Assignment**
>
> We also need to read and return the spacing for Nifti images.
>
> **Add a new feature to `read_nifti`**: Make the function return both the data and the spacing, just like you did for `read_dicom`. Use the nibabel function [nifti_image.header.get_zooms()](https://nipy.org/nibabel/nibabel_images.html#the-image-header) to obtain the spacing.
>
> **Run the code in the second cell below**. *Does it look the same as above?* The code in this cell should work without any modifications. If it doesn't, revise your (earlier?) changes to `read_nifti` accordingly.

In [ ]:
data, spacing = read_nifti('images/ct_jaw.nii.gz')
view(data, spacing=spacing, orientation='sag')

**Supplementary information**: *Pixel spacing and RAS/LPS image orientation are, in fact, simplifications. Both DICOM and Nifti also define an [affine transformation](https://nipy.org/nibabel/coordinate_systems.html#the-affine-matrix-as-a-transformation-between-spaces) that maps pixel coordinates (an index into a matrix) to patient coordinates (a position in mm, with respect to some reference). The definitions RAS and LPS apply to these patient coordinates. In DICOM, each image slice defines its own mapping through the [Image Orientation (Patient)](https://dicom.innolitics.com/ciods/ct-image/image-plane/00200037) and [Image Position (Patient)](https://dicom.innolitics.com/ciods/ct-image/image-plane/00200032) the, from which we need to compute the 3D affine matrix.*

*The axes do not have to be orthogonal, meaning that voxels are no longer 'box-shaped', but rather '3D parallellogram-shaped'. This happens e.g. in CT scans with gantry tilt. In this case, the volume of a voxel should be calculated by [taking the determinant](https://www.euclideanspace.com/maths/geometry/elements/determinant/index.htm) of the top-left 3×3 submatrix of the affine matrix.*

*In this assignment, however, we just assume:*
$$
\mathbf{A} =
\begin{bmatrix}
s_x &   0 &   0 & 0 \\
  0 & s_y &   0 & 0 \\
  0 &   0 & s_z & 0 \\
  0 &   0 &   0 & 1
\end{bmatrix}
$$
*where $(s_x, s_y, s_z)$ is the pixel spacing. This means that the data axes align with the patient axes, and the offset is $(0, 0, 0)$.*

> **Assignment**
>
> Now we have finished our `read_dicom` and `read_nifti` functions, let's add a new function to write Nifti files.
>
> **Implement a `write_nifti` function**: It should take the image data, the spacing, and a file path. In the function, use [nib.nifti1.Nifti1Image(data, affine)](https://nipy.org/nibabel/reference/nibabel.nifti1.html#nifti1image) to create a new nibabel image, and save it with `nib.nifti1.save`. Check your function with the code in the two cells below; it should work without any modifications.
>
> *Hint 1: The `Nifti1Image` requires a 4×4 affine matrix: you can initialize a 4×4 identity matrix with `np.eye(4)`.*
>
> *Hint 2: Think about the ordering and orientation!*

In [ ]:
data, spacing = read_dicom('images/ct_jaw')  # Read 3D DICOM series
write_nifti(data, spacing, 'test.nii.gz')  # Write it to Nifti
read_data, read_spacing = read_nifti('test.nii.gz')
view(read_data, spacing=read_spacing, orientation='sag')

In [ ]:
data, spacing = read_dicom('images/ct_jaw_slice.dcm')  # Read 2D DICOM image
write_nifti(data, spacing, 'test2d.nii.gz')  # Write it to Nifti
read_data, read_spacing = read_nifti('test2d.nii.gz')
view(read_data, spacing=read_spacing)

> **Assignment**
>
> Let's create a segmentation and save it as a Nifti file!
>
> **Run the code below**. *Why doesn't `write_nifti` work for this image?*
>
> **Fix the `write_nifti` function**: The code in the cell below should work without any modifications, while `write_nifti` function should continue to work for the cell above as well!
>
> *Hint: Use Numpy's [astype](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.astype.html) function when necessary.*
>
> Open `test.nii.gz` in ITK-SNAP and add `segmentation.nii.gz` as a segmentation. You can simply drag and drop the first image into ITK-SNAP and select *Load as Main Image*. Then, drag and drop the second image and select *Load as Segmentation*.

In [ ]:
data, spacing = read_dicom('images/ct_jaw')
segmentation = data > 600
view(segmentation, spacing=spacing, orientation='sag')
write_nifti(segmentation, spacing, 'segmentation.nii.gz')